# **This notebook implements a EEG Auditory Attention Detectiom (AAD) system using without using Domain Adversarial Learning. The goal is to predict which speaker a subject is attending to while learning subject-invariant features through domain adaptation.**

# **Importing libraries**

In [1]:
# Standard Library Imports
import os          # File and directory operations
import math        # Mathematical functions
import shutil      # High-level file operations (copy, move, delete)

# Data Handling & Processing
import h5py        # Reading/writing HDF5 (.h5) files
import numpy as np # Numerical computations and arrays
import pandas as pd # Data manipulation and analysis

# Visualization & Progress
import matplotlib.pyplot as plt # Plotting and visualization
from tqdm import tqdm           # Progress bars for loops

# PyTorch: Deep Learning
import torch                      # Core PyTorch library
import torch.nn as nn             # Neural network modules
import torch.optim as optim       # Optimizers (Adam, AdamW, etc.)
from torch.utils.data import DataLoader, Dataset # Dataset and DataLoader utilities


# **Function to load h5 files**

In [2]:
def load_h5_dataset(file_path):
    """
    Load EEG data and labels from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data' and 'label' datasets.

    Returns
    -------
    data : np.ndarray
        EEG data array of shape (N, channels, time_points), where N is the number of samples.
    label : np.ndarray
        Corresponding labels for each EEG sample.

    Notes
    -----
    - Uses h5py to safely read the HDF5 file in read-only mode.
    - Converts HDF5 datasets to NumPy arrays for easier processing downstream.
    """

    # Open HDF5 file in read-only mode
    with h5py.File(file_path, 'r') as f:
        data = np.array(f['data'])   # Load EEG data
        label = np.array(f['label']) # Load labels

    return data, label


# **Define pytorch class for data**

In [3]:
class CustomDatasets(Dataset):
    """
    PyTorch Dataset for EEG data and corresponding event labels.

    This class wraps EEG data and labels so that they can be easily used
    with PyTorch DataLoader for training or evaluation.
    """

    def __init__(self, data, event_data):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like or np.ndarray
            Input EEG data of shape (N, channels, time_points), where N is the number of samples.
        event_data : array-like or np.ndarray
            Labels or event information corresponding to each EEG sample.
        """
        self.data = data
        self.label = event_data

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.label)

    def __getitem__(self, index):
        """
        Retrieve a single sample (data and label) as PyTorch tensors.

        Parameters
        ----------
        index : int
            Index of the sample to retrieve.

        Returns
        -------
        data : torch.FloatTensor
            EEG data for the given sample.
        label : torch.LongTensor
            Corresponding label for the sample.
        """
        # Convert NumPy arrays to PyTorch tensors
        data = torch.Tensor(self.data[index])
        label = torch.LongTensor(self.label[index])

        return data, label


# **Model architecture**

In [4]:
class TokenEmbedding(nn.Module):
    """
    Token embedding module for EEG signals.

    Converts raw EEG input into a higher-dimensional feature representation
    suitable for sequential models like LSTM.
    """

    def __init__(self, c_in, d_model):
        """
        Initialize the TokenEmbedding module.

        Parameters
        ----------
        c_in : int
            Number of EEG channels (input channels).
        d_model : int
            Desired embedding dimension for the output tokens.
        """
        super(TokenEmbedding, self).__init__()

        # First convolution block: increases channel dimension, extracts local features
        self.embed_layer = nn.Sequential(
            nn.Conv2d(1, d_model * 4, kernel_size=(1, 8), padding='same'),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Second convolution block: reduces to desired embedding dimension
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(d_model * 4, d_model, kernel_size=(c_in, 1), padding='valid'),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        Forward pass of TokenEmbedding.

        Parameters
        ----------
        x : torch.Tensor
            Input EEG data of shape (B, channels, time_points)

        Returns
        -------
        torch.Tensor
            Embedded representation of shape (B, time_steps, d_model)
        """
        x = x.unsqueeze(1)             # Add channel dimension for Conv2d (B, 1, C, T)
        x = self.embed_layer(x)        # First conv block
        x = self.embed_layer2(x).squeeze(2)  # Second conv block, remove spatial dim
        x = x.permute(0, 2, 1)         # (B, T, d_model) for LSTM input
        return x


class DARNet_LSTM(nn.Module):
    """
    DARNet LSTM model for EEG classification.

    Consists of:
    - Token embedding module to extract features from EEG
    - Bidirectional LSTM to capture temporal dependencies
    - Linear classifier for final task prediction
    """

    def __init__(self, c_in=32, d_model=16, hidden=64, num_classes=2):
        """
        Initialize DARNet_LSTM model.

        Parameters
        ----------
        c_in : int
            Number of EEG channels (input channels).
        d_model : int
            Dimension of token embeddings.
        hidden : int
            Hidden size of the LSTM.
        num_classes : int
            Number of output classes for classification.
        """
        super().__init__()
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM to capture temporal patterns
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Linear classifier for task prediction
        self.classifier = nn.Linear(hidden * 2, num_classes)  # hidden*2 due to bidirectional LSTM

    def forward(self, x):
        """
        Forward pass of DARNet_LSTM.

        Parameters
        ----------
        x : torch.Tensor
            Input EEG data of shape (B, channels, time_points)

        Returns
        -------
        torch.Tensor
            Output logits of shape (B, num_classes)
        """
        x = self.token_embed(x)        # Embed tokens (B, T, d_model)
        x, _ = self.lstm(x)            # LSTM outputs (B, T, hidden*2)
        x = x.mean(dim=1)              # Global average pooling over time dimension
        out = self.classifier(x)       # Class logits
        return out


# **Function for training**

In [5]:
def train_model(model, train_loader, val_loader, epochs=100, lr=5e-4, weight_decay=3e-4, device="cuda"):
    """
    Train a PyTorch model with validation monitoring and automatic best-model saving.
    """

    # Move model to GPU/CPU
    model = model.to(device)

    # Loss function for 2-class classification
    criterion = nn.CrossEntropyLoss()

    # AdamW optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Cosine LR scheduler for smooth learning rate decay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=10, eta_min=1e-6
    )

    best_val_acc = 0.0  # Track best validation score

    # Lists to store training curves (optional)
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        
        #   TRAINING PHASE
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for x, y in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            x = x.to(device)
            y = y.to(device).long().squeeze(-1)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            # Track metrics
            running_loss += loss.item() * x.size(0)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        #   VALIDATION PHASE
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}"):
                x = x.to(device)
                y = y.to(device).long().squeeze(-1)

                logits = model(x)
                loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        #   LOG PROGRESS
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

        #   SAVE BEST MODEL
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print("Saved new best model")

        scheduler.step()

    print("Training complete.")
    print(f"Best validation accuracy: {best_val_acc:.4f}")

    return model


# **Load the data into training and val loaders**

In [6]:
# File paths for training and validation HDF5 datasets
train_h5_path = '/kaggle/input/train_data.h5'
val_h5_path   = '/kaggle/input/val_data.h5'

# Load EEG data and labels from HDF5 files
X_train, y_train = load_h5_dataset(train_h5_path)  # Training data
X_val, y_val     = load_h5_dataset(val_h5_path)    # Validation data

# Print shapes to verify data loading
print(f"Train data: {X_train.shape}, labels: {y_train.shape}")
print(f"Val data:   {X_val.shape}, labels: {y_val.shape}")

# Wrap NumPy arrays into PyTorch Dataset objects
train_dataset = CustomDatasets(X_train, y_train)
val_dataset   = CustomDatasets(X_val, y_val)

# Build DataLoaders for batching during training and validation
train_loader = DataLoader(
    train_dataset,
    batch_size=128,   # Number of samples per batch
    shuffle=True,     # Shuffle training data for randomness
    drop_last=True    # Drop last incomplete batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,    # No need to shuffle validation data
    drop_last=False   # Keep all validation samples
)


Train data: (171080, 32, 128), labels: (171080, 1)
Val data:   (26320, 32, 128), labels: (26320, 1)


# **Train the model**

In [7]:
# Create the model
model = DARNet_LSTM(d_model = 8)
# Run the training
model = train_model(model, train_loader, val_loader, epochs=200, lr=1e-4, weight_decay = 3e-4, device="cuda")

Training Epoch 1:   0%|          | 0/1336 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return F.conv2d(
Validation Epoch 1: 100%|██████████| 206/206 [00:01<00:00, 159.28it/s]


Epoch 1/200 | Train Loss: 0.5992 | Train Acc: 0.6518 | Val Loss: 1.0516 | Val Acc: 0.4849 | LR: 0.000100
Saved new best model


Validation Epoch 2: 100%|██████████| 206/206 [00:01<00:00, 161.66it/s]


Epoch 2/200 | Train Loss: 0.5224 | Train Acc: 0.7212 | Val Loss: 1.0844 | Val Acc: 0.4940 | LR: 0.000098
Saved new best model


Validation Epoch 3: 100%|██████████| 206/206 [00:01<00:00, 160.25it/s]


...

Epoch 198/200 | Train Loss: 0.1007 | Train Acc: 0.9640 | Val Loss: 3.1510 | Val Acc: 0.4350 | LR: 0.000080


Validation Epoch 199: 100%|██████████| 206/206 [00:01<00:00, 158.54it/s]


Epoch 199/200 | Train Loss: 0.1061 | Train Acc: 0.9626 | Val Loss: 3.0854 | Val Acc: 0.4298 | LR: 0.000091


Validation Epoch 200: 100%|██████████| 206/206 [00:01<00:00, 158.72it/s]

Epoch 200/200 | Train Loss: 0.1049 | Train Acc: 0.9626 | Val Loss: 2.8564 | Val Acc: 0.4189 | LR: 0.000098
Training complete.
Best validation accuracy: 0.4940
